# Immune Data Pretraining


In [16]:
from datasets import load_from_disk
import torch
from torch.utils.data import DataLoader

from trainer import ScImmunePretrainingTrainer, ScImmuneTrainingArguments

from model import ScImmuneModel
from config import ScImmuneConfig

from accelerate import notebook_launcher

from tokenizer import ScImmuneTokenizer
from collator import ScImmuneDataCollator

In [3]:
tokenizer = ScImmuneTokenizer(vocab_file="vocab_with_metadata.json")

In [4]:
## Set data folder
tokenized_data_path = 'scimmune-model/tokenized_data'

## Load dataset
tokenized_dataset = load_from_disk(tokenized_data_path)

### Post-processing tokenized output

In [5]:
# Number of metadata tokens you want to add zeros for
num_metadata_tokens = 6


def prepend_zeros(example):
    zeros = torch.zeros(num_metadata_tokens, dtype=example["values"].dtype)
    example["values"] = torch.cat((zeros, example["values"]))
    return example

# Map over dataset
tokenized_dataset = tokenized_dataset.map(prepend_zeros)

In [7]:
tokenized_dataset[0]

{'genes': tensor([60801, 60926, 60942, 61043, 61046, 61045, 60695,  4441,  6019,  8023,
          3933,  2114,  3573,  4452, 15664,  3341, 16375, 30442, 11130, 60697,
         20091,  9801, 20797, 16339, 10113, 18757, 32057, 21084,  4029,  5216,
         34889, 32342,  5254, 20881, 21348, 30333,  4453,  8712,  7702,  1448,
         20880,  3708, 11047, 19063, 18166, 19580, 19850, 16237,  8200, 16406,
         33798,  8149,  8799,  9567, 35766, 11015,  3985, 12916,  7867,  8356,
         18305,  9672, 35798, 60697, 20316, 34072, 18889, 35668, 31856, 10632,
         17603, 19165, 36514, 19205,  4378,  4379, 12828, 19593, 16871, 35305,
          5242,  1519, 17211,  3834,  5106,  8732, 31534,  8376, 35021, 32258,
         21424, 12146, 18900,  4917, 60697, 20516,  1878, 30350, 34488,  2936,
          2581, 33868,  3452,  7854, 10948, 33996, 18858, 30316,  8334, 33878,
         30431, 31983, 21071, 20109, 32368, 34605, 21440, 32588, 30686, 35739,
         12392, 16647, 12352, 30308, 30370,

In [8]:
cls_token_id = 60695  # ID for <cls>

def move_cls_to_front(example):
    genes = example["genes"].tolist()   # convert tensor → list

    if cls_token_id in genes:
        idx = genes.index(cls_token_id)

        # Move CLS to the front
        genes.pop(idx)

        genes.insert(0, cls_token_id)

    # Convert back to tensors for HF dataset
    example["genes"] = torch.tensor(genes, dtype=torch.long)

    return example

tokenized_dataset = tokenized_dataset.map(move_cls_to_front)

In [9]:
tokenized_dataset = tokenized_dataset.rename_column("values", "expressions")
tokenized_dataset[0]

{'genes': tensor([60695, 60801, 60926, 60942, 61043, 61046, 61045,  4441,  6019,  8023,
          3933,  2114,  3573,  4452, 15664,  3341, 16375, 30442, 11130, 60697,
         20091,  9801, 20797, 16339, 10113, 18757, 32057, 21084,  4029,  5216,
         34889, 32342,  5254, 20881, 21348, 30333,  4453,  8712,  7702,  1448,
         20880,  3708, 11047, 19063, 18166, 19580, 19850, 16237,  8200, 16406,
         33798,  8149,  8799,  9567, 35766, 11015,  3985, 12916,  7867,  8356,
         18305,  9672, 35798, 60697, 20316, 34072, 18889, 35668, 31856, 10632,
         17603, 19165, 36514, 19205,  4378,  4379, 12828, 19593, 16871, 35305,
          5242,  1519, 17211,  3834,  5106,  8732, 31534,  8376, 35021, 32258,
         21424, 12146, 18900,  4917, 60697, 20516,  1878, 30350, 34488,  2936,
          2581, 33868,  3452,  7854, 10948, 33996, 18858, 30316,  8334, 33878,
         30431, 31983, 21071, 20109, 32368, 34605, 21440, 32588, 30686, 35739,
         12392, 16647, 12352, 30308, 30370,

## Collator

In [10]:
ds = tokenized_dataset.with_format(type="torch", columns=["genes","expressions"])

In [11]:
pad_token_id_value = 60694

collator = ScImmuneDataCollator(
    do_padding=True,
    pad_token_id=pad_token_id_value,  # match your vocab
    pad_value=-2,
    do_mlm=True,
    do_binning=False,       # applies  preprocess.binning (51 bins in your code)
    mlm_probability=0.15,
    mask_value=-1,
    max_length=2000,
    sampling=True,
    keep_first_n_tokens=1 + num_metadata_tokens,  # <cls> + metadata are preserved
    data_style="both",     # "pcpt" or "both" are typical for pretraining
)

### Collator Testing

In [ ]:
pad_id = pad_token_id_value
keep = 1 + num_metadata_tokens
n_bins = 51

# quick sanity check
from torch.utils.data import DataLoader
dl = DataLoader(ds, batch_size=8, shuffle=False, collate_fn=collator)
batch = next(iter(dl))
for k, v in batch.items():
    print(k, v.shape, v.dtype)

In [ ]:
# 4.1 shapes match
assert batch["pcpt_gene"].shape == batch["pcpt_expr"].shape == batch["masked_expr"].shape

# 4.2 padding behavior
assert (batch["expr"][batch["gene"] == pad_id] == 0).all()        # expr pad_value
assert (batch["gene"] == pad_id).any() or True                     # padding exists or not

# 4.3 prefix untouched (no masking/binning)
# masked_expr must equal expr in prefix; prefix should be zeros if you set them that way
assert torch.equal(batch["masked_expr"][:, :keep], batch["expr"][:, :keep])

# 4.4 post-prefix binned range
post = batch["expr"][:, keep:]
assert torch.isfinite(post).all()
assert (post >= 0).all() & (post < n_bins).all()
assert torch.allclose(post, post.round())  # integer-like after binning

In [ ]:
row = 0
prefix_ids = batch["gene"][row, :keep].tolist()
prefix_tokens = tokenizer.convert_ids_to_tokens(prefix_ids)
print("Prefix tokens:", prefix_tokens)  # expect: ['<cls>', '<cell_type=...>', '<tissue=...>', ...]

print("Prefix expr :", batch["expr"][row, :keep].tolist())         # expect zeros (or unbinned if you chose)
print("Prefix masked:", batch["masked_expr"][row, :keep].tolist()) # should match expr (no -1 here)

## Dataloader

In [12]:
# Load data

dset = ds.train_test_split(test_size=0.02, seed=42, shuffle=True)  # tiny val split

train_dataset = dset["train"]
valid_dataset = dset["test"]

## Config and Model

In [13]:
cfg = ScImmuneConfig.from_pretrained("scImmune_metadata_model/")
model = ScImmuneModel.from_pretrained("scImmune_metadata_model/", config=cfg)
model

ScImmuneModel(
  (gene_encoder): GeneEncoder(
    (embedding): Embedding(61048, 512, padding_idx=60694)
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (value_encoder): ContinuousValueEncoder(
    (lin1): Linear(in_features=1, out_features=512, bias=True)
    (act): ReLU()
    (lin2): Linear(in_features=512, out_features=512, bias=True)
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (drop): Dropout(p=0.0, inplace=False)
  )
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-11): 12 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=512, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (linear2): Linear(in_features=512, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
       

## Trainer

In [14]:
training_args = ScImmuneTrainingArguments(
    output_dir="runs/scimmune-ctpt",
    per_device_train_batch_size=8,          # adjust by GPU mem
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,          # keeps effective batch size up
    learning_rate=1e-4,                     # low LR for continual PT
    weight_decay=0.01,
    max_steps=50000,                       # or num_train_epochs=3
    lr_scheduler_type="cosine",
    warmup_ratio=0.10,                      # 10% warmup
    logging_steps=50,
    save_steps=1000,
    eval_steps=1000,
    evaluation_strategy="steps",
    save_total_limit=3,
    fp16=True,                              # or bf16=True on Ampere+
    dataloader_num_workers=4,
    seed=42,

    # scGPT-specific fields your Trainer/Collator read:
    mlm_probability=0.15,                   # keep at 0.15 for stability
    max_length=2000,                       
    MVC=False                               # enable later if you need it
)

In [15]:
trainer = ScImmunePretrainingTrainer(
        model=model,
        args=training_args,
        data_collator=collator,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
    )

In [17]:
def train_distributed():
    trainer.train()
    trainer.save_model("runs/scimmune-ctpt/final")

In [18]:
notebook_launcher(train_distributed, args=(), num_processes=3, mixed_precision="fp16")

ValueError: To launch a multi-device training from your notebook, the `Accelerator` should only be initialized inside your training function. Restart your notebook and make sure no cells initializes an `Accelerator`.